# Your Details

Your Name: Aashrith Sai Yamsani

Your ID Number: 25287419

# Etivity Task 4 - Part 1: Pruning a TensorFlow/Keras Model exercise

For this exercise, you will prune a simple convolutional neural network (CNN) trained on the Fashion MNIST dataset. The first section of this exercise is already completed (Parts 1-5), but you will have to run the cells. Your task is to perform pruning on this model uses the TF Model optimisations toolkit and report on the results.


By the end of this notebook, you'll be able to:

* Understand Pruning in TensorFlow
* Prune a basic CNN using the TensorFlow Model optimisation framework
* Analyse the model perfromance
* Results analysis

**Start** with the design in sections [1], [2], [3], [4] and [5] for which code is provided - then proceed to section [6] to begin this model pruning exercise.

    [1] Import data dependencies
    [2] Load the Fashion MNIST dataset
    [3] Prepocess the data
    [4] Create and train the CNN model
    [5] Evaluate the model performance
    [6] Prune the network and analyse your results
    
   
### Important Note 1 on Submission

There are code exercises to complete in this task.  Insert your code entries into the cell areas marked with the 'enter code here' text as below, so that grading can easily be assessed.

\### **ENTER CODE HERE**

Please make sure you are not doing the following:

1. You have not added any _extra_ `print` statement(s) in the assignment.
2. You have not added any _extra_ code cell(s) in the assignment.
3. You have not changed any of the function parameters.
4. You are not using any global variables inside your graded exercises. Unless specifically instructed to do so, please refrain from it and use the local variables instead.
5. You are not changing the assignment code where it is not required, like creating _extra_ variables.

### Important Note 2 on Submission

There are <font color='red'>**DISCUSSION**</font> elements to include at the end of this notebook. Make sure to read the notes carefully and answer the questions asked. Include code where it is requested in order to get good marks


### Let's get started!


### Installing the TensorFlow Model Optimisation toolkit

You must first install it using pip (No need to do this as already installed, hence commented out).

In [1]:
#pip install --user --upgrade tensorflow-model-optimization

In [2]:
#pip install tf-keras

## 1. Import the data dependencies

In [ ]:
import tensorflow as tf
import tf_keras as keras
from keras.datasets import fashion_mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.utils import to_categorical
import tensorflow_model_optimization as tfmot

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tempfile
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
%matplotlib inline

## 2. Load the dataset
We will use the Fashion-MNIST dataset and view a random set of images from the dataset.

In [ ]:
# Load Fashion MNIST dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

Create a dictionary of all classes in the target - Note there are 10 classes in this dataset.

In [ ]:
# Map for human readable class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
class_labels = pd.Series(['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Code', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot'])
labels_dict = class_labels.to_dict()
labels_dict

Get 9 images at random from the training data, plot and fetch their corresponding labels from the training targets.

In [ ]:
np.random.seed(11)
plt.figure(figsize=(10, 10))
for i, rand_num in enumerate(np.random.randint(0, len(X_train), 9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[rand_num]), plt.axis('off')
    plt.title(labels_dict[y_train[rand_num]])
    plt.axis("off")
plt.show()

## 3. Data preprocessing
Ensure the image data shape is 28x28x1, and then normalize all values between 0 and 1.

In [ ]:
# Model configuration
img_width, img_height = 28, 28
no_classes = 10

# Reshape data for CNN
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1)
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1)

input_shape = (img_width, img_height, 1)

# Parse numbers as floats
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# Normalize [0, 255] into [0, 1]
X_train = X_train / 255
X_test = X_test / 255

# Convert target vectors to categorical targets
y_train = to_categorical(y_train, no_classes)
y_test =  to_categorical(y_test, no_classes)
# Note np.argmax(y_test, axis=-1) returns original digit array

Finally, we create a validation dataset using train_test_split()

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.16666)

print('Shape of data used for training, and shape of training targets : \n ', X_train.shape, ',', y_train.shape)
print('Shape of data used for validation, and shape of validation targets: \n ', X_valid.shape, ',', y_valid.shape)
print('Shape of data used for test, and shape of test targets: \n ', X_test.shape, ',', y_test.shape)

## 4. Create and train the model

In [ ]:
# Create the model
model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Flatten(),
  keras.layers.Dense(256, activation='relu'),
  keras.layers.Dense(no_classes, activation='softmax')
])

# Compile the model
model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer='adam',
              metrics=['accuracy'])

batch_size = 32
no_epochs = 12

# Fit data to model
history = model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=no_epochs,
          verbose=1,
          validation_data=(X_valid,y_valid))

## 5. Evaluate the model performance

In [ ]:
# Plot the performance
plt.figure(figsize=(6, 5))
plt.plot(history.history['accuracy'], color='r')
plt.plot(history.history['val_accuracy'], color='b')
plt.title('Model Accuracy', weight='bold', fontsize=16)
plt.ylabel('accuracy', weight='bold', fontsize=14)
plt.xlabel('epoch', weight='bold', fontsize=14)
plt.ylim(0.5, 1)
plt.xticks(weight='bold', fontsize=12)
plt.yticks(weight='bold', fontsize=12)
plt.legend(['train', 'val'], loc='upper left', prop={'size': 14})
plt.grid(color = 'y', linewidth='0.5')
plt.show()

In [ ]:
# Get Model Predictions for test data
predictions = model.predict(X_test)
print(classification_report(np.argmax(y_test, axis=-1), np.argmax(predictions, axis=-1),target_names=class_labels))

### Test the model accuracy

In [ ]:
# Generate generalization metrics
score = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')

### Save the model
Make sure to store your model to a temporary file, so that you can compare the sizes of the original and the pruned model later:

In [ ]:
# Store file
_, keras_file = tempfile.mkstemp('.h5')
tf.keras.models.save_model(model, keras_file, include_optimizer=False)
print('Saved baseline model to:', keras_file)

## 6. Pruning exercise

**Include your code in the cells below** where it states **### ENTER CODE HERE**

### Configure the pruning process

- Load functionality for adding pruning wrappers to make sure the model's layers are prunable.
- Set the pruning configuration with the following:
   1. Load the number of images used in the training set.
   2. Compute the *end_step* of the pruning process using batch size, the number of images and the number of epochs.
   3. Define the pruning operation using **pruning_params**. Initially set the model to be 50% sparse (50% zeros in weights) increasing to 80%. Begin at 0 and end at *end_step*.
   4. Call the **prune_low_magnitude** functionality to generate the prunable model from the initial model and the defined **pruning_params**.

In [3]:
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

num_images = X_train.shape[0]

end_step = np.ceil(num_images / batch_size).astype(np.int32) * no_epochs

pruning_params = {'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,final_sparsity=0.80,begin_step=0,end_step=end_step)}

model_for_pruning = prune_low_magnitude(model, **pruning_params)

model_for_pruning.summary()

SyntaxError: invalid syntax (2577122249.py, line 2)

### Start the pruning process
After configuring the pruning process, you need to recompile the model and start the pruning process. Use the **UpdatePruningStep** callback here, because it propagates optimizer activities to the pruning process.

In [ ]:
### ENTER CODE HERE
model_for_pruning.compile(optimizer='adam',loss=tf.keras.losses.categorical_crossentropy,metrics=['accuracy'])

callbacks = [tfmot.sparsity.keras.UpdatePruningStep(),]

history_pruned = model_for_pruning.fit(X_train, y_train,batch_size=batch_size,epochs=no_epochs,validation_data=(X_valid, y_valid),callbacks=callbacks)

### Measure the pruning effectiveness

- By measuring how much the performance has changed, compared to before pruning;
- By measuring how much the model size has changed, compared to before pruning.


In [ ]:
### ENTER CODE HERE
score_pruned = model_for_pruning.evaluate(X_test, y_test, verbose=0)
print(f'Pruned model Test loss: {score_pruned[0]} / Test accuracy: {score_pruned[1]}')

### Save/export the pruned model

In [ ]:
### ENTER CODE HERE

### Compare the model sizes

How much smaller is the pruned model size?

In [ ]:
### ENTER CODE HERE

## Combining Pruning with Quantization for more optimization
It is possible to make the model even smaller by applying **quantization**.
Adding quantization first requires you to add a TFLite converter. This converter converts your TensorFlow model into TensorFlow Lite equivalent, which is what quantization will run against. Converting the model into a Lite model allows us to specify a model optimizer - use the DEFAULT or dynamic range quantization for this exercise.

In [ ]:
### ENTER CODE HERE

### <span style='color: red;'>Discuss you results:  How smaller are the model sizes?</span> ###

What are the model sizes?
- What is the size improvement for original --> pruning ?
- What is the size improvement for pruning --> quantization?
- What is the total size improvement for pruning + quantization?

What are the differences in model accuracies?
- Original model accuracy?
- Pruned model accuracy?
- Pruned and Quantized model accuracy?

*Include you final comments here*

### <span style='color: red;'>Discuss what other pruning strategies you could try for the original model?</span> ###

#### Can you show any code to demonstate this and discuss/compare the results further?

In [ ]:
### Insert any code here